# Scenario Analysis Notebook

This notebook analyzes the pre-injected scenarios in the SAP mock data.

## Scenarios Included

| ID | Scenario | Description | Key Data |
|-----|----------|-------------|----------|
| SCN1 | Blocked Stock Deviation | Inventory discrepancy discovered during cycle count | BWART 344 |
| SCN2 | Product Contamination | Batch scrapped due to contamination | MAT-A0005, BWART 551 |
| SCN3 | Warehouse Fire | Stock destroyed by fire at Plant 2000 | BWART 551 |
| SCN8 | Temperature Excursion | Product moved to QA hold after cold chain breach | BWART 311 |
| SCN9 | Rerouted Shipment | Goods received at alternate location | BWART 101 |
| - | India Customer | Dedicated customer/product relationship | CUST00020 + MAT-A0020 |

In [ ]:
# CELL: CONFIGURATION
# Set these to match your data location
CATALOG = "sample_synthetic_sap"
SCHEMA = "sap"

print(f"Analyzing scenarios in: {CATALOG}.{SCHEMA}")

In [ ]:
# CELL: LOAD TABLES
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Load all relevant tables
df_matdoc = spark.table(f"{CATALOG}.{SCHEMA}.matdoc")
df_vbak = spark.table(f"{CATALOG}.{SCHEMA}.vbak")
df_vbap = spark.table(f"{CATALOG}.{SCHEMA}.vbap")
df_likp = spark.table(f"{CATALOG}.{SCHEMA}.likp")
df_lips = spark.table(f"{CATALOG}.{SCHEMA}.lips")
df_mara = spark.table(f"{CATALOG}.{SCHEMA}.mara")
df_makt = spark.table(f"{CATALOG}.{SCHEMA}.makt")
df_kna1 = spark.table(f"{CATALOG}.{SCHEMA}.kna1")
df_mbew = spark.table(f"{CATALOG}.{SCHEMA}.mbew")

print("Tables loaded successfully")

---
## 1. Scenario Overview
View all injected scenario records from MATDOC

In [ ]:
# CELL: SCENARIO OVERVIEW
# Find all scenario records (identified by MBLNR starting with 'SCN')

df_scenarios = (
    df_matdoc
    .filter(F.col("MBLNR").startswith("SCN"))
    .join(df_makt.filter(F.col("SPRAS") == "EN"), "MATNR", "left")
    .select(
        F.col("MBLNR").alias("Scenario_ID"),
        F.col("BKTXT").alias("Description"),
        F.col("MATNR").alias("Material"),
        F.col("MAKTX").alias("Material_Desc"),
        F.col("WERKS").alias("Plant"),
        F.col("LGORT").alias("Storage_Loc"),
        F.col("BWART").alias("Mvmt_Type"),
        F.col("SHKZG").alias("Debit_Credit"),
        F.col("MENGE").alias("Quantity"),
        F.col("BUDAT").alias("Posting_Date")
    )
    .orderBy("Scenario_ID")
)

print("=" * 70)
print("INJECTED SCENARIOS SUMMARY")
print("=" * 70)
display(df_scenarios)

---
## 2. SCN2 - Product Contamination Analysis (MAT-A0005)
Analyze the impact of the contamination/recall scenario

In [ ]:
# CELL: CONTAMINATION ANALYSIS
CONTAMINATED_MATERIAL = "MAT-A0005"

print("=" * 70)
print(f"CONTAMINATION SCENARIO ANALYSIS: {CONTAMINATED_MATERIAL}")
print("=" * 70)

# Get material details
material_info = (
    df_mara.filter(F.col("MATNR") == CONTAMINATED_MATERIAL)
    .join(df_makt.filter(F.col("SPRAS") == "EN"), "MATNR", "left")
    .select("MATNR", "MAKTX", "MTART", "MATKL")
    .first()
)

if material_info:
    print(f"\nMaterial: {material_info['MATNR']}")
    print(f"Description: {material_info['MAKTX']}")
    print(f"Type: {material_info['MTART']}")

# Scrap quantity from scenario
scrap_record = df_matdoc.filter(
    (F.col("MBLNR") == "SCN002") & (F.col("MATNR") == CONTAMINATED_MATERIAL)
).first()

if scrap_record:
    print(f"\nSCRAP EVENT:")
    print(f"  Quantity Scrapped: {scrap_record['MENGE']:,.0f} units")
    print(f"  Movement Type: {scrap_record['BWART']} (Dangerous Process Scrap)")
    print(f"  Plant: {scrap_record['WERKS']}")
    print(f"  Date: {scrap_record['BUDAT']}")

# Get standard cost for financial impact
cost_info = df_mbew.filter(F.col("MATNR") == CONTAMINATED_MATERIAL).first()
if cost_info and scrap_record:
    financial_impact = float(scrap_record['MENGE']) * float(cost_info['STPRS'])
    print(f"\nFINANCIAL IMPACT:")
    print(f"  Standard Cost: £{float(cost_info['STPRS']):,.2f} per unit")
    print(f"  Total Write-Off: £{financial_impact:,.2f}")

In [ ]:
# CELL: CONTAMINATED MATERIAL - SALES IMPACT
# Check sales orders affected by this material

print("\n" + "=" * 70)
print(f"SALES ORDERS CONTAINING {CONTAMINATED_MATERIAL}")
print("=" * 70)

df_affected_orders = (
    df_vbap.filter(F.col("MATNR") == CONTAMINATED_MATERIAL)
    .join(df_vbak, "VBELN", "inner")
    .join(df_kna1, df_vbak["KUNNR"] == df_kna1["KUNNR"], "left")
    .groupBy(df_kna1["KUNNR"], "NAME1")
    .agg(
        F.count("VBELN").alias("Order_Count"),
        F.sum("KWMENG").alias("Total_Qty_Ordered"),
        F.sum("NETWR").alias("Total_Value")
    )
    .orderBy(F.col("Total_Value").desc())
)

print(f"\nCustomers who ordered {CONTAMINATED_MATERIAL}:")
display(df_affected_orders)

# Summary stats
totals = df_affected_orders.agg(
    F.sum("Order_Count").alias("Total_Orders"),
    F.sum("Total_Qty_Ordered").alias("Total_Units"),
    F.sum("Total_Value").alias("Total_Revenue")
).first()

print(f"\nPOTENTIAL RECALL IMPACT:")
print(f"  Orders to Review: {totals['Total_Orders']:,.0f}")
print(f"  Units Shipped: {totals['Total_Units']:,.0f}")
print(f"  Revenue at Risk: £{totals['Total_Revenue']:,.2f}")

---
## 3. SCN3 - Warehouse Fire Analysis (Plant 2000)
Analyze the stock loss from the warehouse fire

In [ ]:
# CELL: WAREHOUSE FIRE ANALYSIS

print("=" * 70)
print("WAREHOUSE FIRE SCENARIO ANALYSIS: Plant 2000")
print("=" * 70)

# Get fire scenario details
fire_record = df_matdoc.filter(F.col("MBLNR") == "SCN003").first()

if fire_record:
    material = fire_record['MATNR']
    
    # Get material description
    mat_desc = df_makt.filter(
        (F.col("MATNR") == material) & (F.col("SPRAS") == "EN")
    ).first()
    
    print(f"\nFIRE EVENT DETAILS:")
    print(f"  Material: {material}")
    print(f"  Description: {mat_desc['MAKTX'] if mat_desc else 'N/A'}")
    print(f"  Plant: {fire_record['WERKS']} (Rotterdam)")
    print(f"  Storage Location: {fire_record['LGORT']}")
    print(f"  Quantity Lost: {fire_record['MENGE']:,.0f} units")
    print(f"  Movement Type: {fire_record['BWART']} (Scrapping)")
    
    # Financial impact
    cost_info = df_mbew.filter(
        (F.col("MATNR") == material) & (F.col("BWKEY") == "2000")
    ).first()
    
    if cost_info:
        loss_value = float(fire_record['MENGE']) * float(cost_info['STPRS'])
        print(f"\nFINANCIAL IMPACT:")
        print(f"  Unit Cost: £{float(cost_info['STPRS']):,.2f}")
        print(f"  Total Loss: £{loss_value:,.2f}")

# Check pending deliveries from Plant 2000
print(f"\n" + "-" * 50)
print("PENDING DELIVERIES FROM PLANT 2000:")

df_pending = (
    df_lips.filter(F.col("WERKS") == "2000")
    .join(df_likp, "VBELN", "inner")
    .groupBy("MATNR")
    .agg(
        F.count("VBELN").alias("Delivery_Count"),
        F.sum("LFIMG").alias("Total_Qty")
    )
    .join(df_makt.filter(F.col("SPRAS") == "EN"), "MATNR", "left")
    .select("MATNR", "MAKTX", "Delivery_Count", "Total_Qty")
    .orderBy(F.col("Total_Qty").desc())
    .limit(10)
)

display(df_pending)

---
## 4. SCN8 - Temperature Excursion Analysis
Analyze the quality hold scenario

In [ ]:
# CELL: TEMPERATURE EXCURSION ANALYSIS

print("=" * 70)
print("TEMPERATURE EXCURSION SCENARIO ANALYSIS")
print("=" * 70)

# Get temperature scenario details
temp_record = df_matdoc.filter(F.col("MBLNR") == "SCN008").first()

if temp_record:
    material = temp_record['MATNR']
    
    mat_desc = df_makt.filter(
        (F.col("MATNR") == material) & (F.col("SPRAS") == "EN")
    ).first()
    
    print(f"\nTEMPERATURE EXCURSION EVENT:")
    print(f"  Material: {material}")
    print(f"  Description: {mat_desc['MAKTX'] if mat_desc else 'N/A'}")
    print(f"  Plant: {temp_record['WERKS']} (Frankfurt)")
    print(f"  From Location: {temp_record['LGORT']}")
    print(f"  To Location: {temp_record['UMLGO']} (QA Hold)")
    print(f"  Quantity on Hold: {temp_record['MENGE']:,.0f} units")
    print(f"  Movement Type: {temp_record['BWART']} (Transfer Posting)")
    
    # Financial exposure
    cost_info = df_mbew.filter(
        (F.col("MATNR") == material) & (F.col("BWKEY") == temp_record['WERKS'])
    ).first()
    
    if cost_info:
        exposure = float(temp_record['MENGE']) * float(cost_info['STPRS'])
        print(f"\nFINANCIAL EXPOSURE (if scrapped):")
        print(f"  Unit Cost: £{float(cost_info['STPRS']):,.2f}")
        print(f"  Potential Loss: £{exposure:,.2f}")
        print(f"\nNote: Stock is on QA hold pending quality investigation.")
        print(f"      May be released if testing passes.")

---
## 5. India Customer Analysis (CUST00020 + MAT-A0020)
Analyze the dedicated customer/product relationship

In [ ]:
# CELL: INDIA CUSTOMER ANALYSIS
INDIA_CUSTOMER = "CUST00020"
INDIA_PRODUCT = "MAT-A0020"

print("=" * 70)
print("INDIA CUSTOMER SCENARIO ANALYSIS")
print("=" * 70)

# Get customer details
customer_info = df_kna1.filter(F.col("KUNNR") == INDIA_CUSTOMER).first()

if customer_info:
    print(f"\nCUSTOMER PROFILE:")
    print(f"  Customer ID: {customer_info['KUNNR']}")
    print(f"  Name: {customer_info['NAME1']}")
    print(f"  City: {customer_info['ORT01']}")
    print(f"  Country: {customer_info['LAND1']}")

# Get product details
product_info = (
    df_mara.filter(F.col("MATNR") == INDIA_PRODUCT)
    .join(df_makt.filter(F.col("SPRAS") == "EN"), "MATNR", "left")
    .first()
)

if product_info:
    print(f"\nDEDICATED PRODUCT:")
    print(f"  Material: {product_info['MATNR']}")
    print(f"  Description: {product_info['MAKTX']}")
    print(f"  Type: {product_info['MTART']}")

In [ ]:
# CELL: INDIA CUSTOMER - ORDER ANALYSIS

print("\n" + "=" * 70)
print("INDIA CUSTOMER ORDER ANALYSIS")
print("=" * 70)

# All orders for India customer
df_india_orders = (
    df_vbak.filter(F.col("KUNNR") == INDIA_CUSTOMER)
    .join(df_vbap, "VBELN", "inner")
)

# Summary by product
df_india_by_product = (
    df_india_orders
    .groupBy("MATNR")
    .agg(
        F.countDistinct("VBELN").alias("Order_Count"),
        F.sum("KWMENG").alias("Total_Qty"),
        F.sum("NETWR").alias("Total_Value")
    )
    .join(df_makt.filter(F.col("SPRAS") == "EN"), "MATNR", "left")
    .select("MATNR", "MAKTX", "Order_Count", "Total_Qty", "Total_Value")
    .orderBy(F.col("Total_Value").desc())
)

print("\nProducts ordered by India Customer:")
display(df_india_by_product)

# Check exclusivity - does anyone else order MAT-A0020?
print("\n" + "-" * 50)
print(f"EXCLUSIVITY CHECK: Who else orders {INDIA_PRODUCT}?")

df_other_customers = (
    df_vbap.filter(F.col("MATNR") == INDIA_PRODUCT)
    .join(df_vbak, "VBELN", "inner")
    .filter(F.col("KUNNR") != INDIA_CUSTOMER)
    .groupBy("KUNNR")
    .agg(
        F.countDistinct("VBELN").alias("Order_Count"),
        F.sum("KWMENG").alias("Total_Qty")
    )
    .join(df_kna1.select("KUNNR", "NAME1"), "KUNNR", "left")
)

other_count = df_other_customers.count()
if other_count == 0:
    print(f"\n✓ {INDIA_PRODUCT} is EXCLUSIVE to {INDIA_CUSTOMER}")
    print(f"  No other customers have ordered this product.")
else:
    print(f"\n⚠ {other_count} other customer(s) also ordered {INDIA_PRODUCT}:")
    display(df_other_customers)

---
## 6. SCN9 - Rerouted Shipment Analysis
Analyze goods received at alternate location

In [ ]:
# CELL: REROUTED SHIPMENT ANALYSIS

print("=" * 70)
print("REROUTED SHIPMENT SCENARIO ANALYSIS")
print("=" * 70)

# Get reroute scenario details
reroute_record = df_matdoc.filter(F.col("MBLNR") == "SCN009").first()

if reroute_record:
    material = reroute_record['MATNR']
    
    mat_desc = df_makt.filter(
        (F.col("MATNR") == material) & (F.col("SPRAS") == "EN")
    ).first()
    
    print(f"\nREROUTED RECEIPT EVENT:")
    print(f"  Material: {material}")
    print(f"  Description: {mat_desc['MAKTX'] if mat_desc else 'N/A'}")
    print(f"  Destination Plant: {reroute_record['WERKS']} (Warsaw)")
    print(f"  Storage Location: {reroute_record['LGORT']} (Alternate)")
    print(f"  Quantity Received: {reroute_record['MENGE']:,.0f} units")
    print(f"  Movement Type: {reroute_record['BWART']} (Goods Receipt)")
    
    print(f"\nIMPLICATIONS:")
    print(f"  - Stock is at alternate location, may need transfer")
    print(f"  - Additional logistics cost may apply")
    print(f"  - Delivery lead times may be impacted")

---
## 7. Scenario Financial Summary
Consolidated view of all scenario impacts

In [ ]:
# CELL: FINANCIAL SUMMARY
import plotly.graph_objects as go

print("=" * 70)
print("SCENARIO FINANCIAL IMPACT SUMMARY")
print("=" * 70)

# Calculate financial impact for each scenario
scenario_impacts = []

for scenario_id in ["SCN001", "SCN002", "SCN003", "SCN008", "SCN009"]:
    record = df_matdoc.filter(F.col("MBLNR") == scenario_id).first()
    if record:
        material = record['MATNR']
        plant = record['WERKS']
        qty = float(record['MENGE'])
        
        # Get cost
        cost_record = df_mbew.filter(
            (F.col("MATNR") == material) & (F.col("BWKEY") == plant)
        ).first()
        
        if not cost_record:
            cost_record = df_mbew.filter(F.col("MATNR") == material).first()
        
        unit_cost = float(cost_record['STPRS']) if cost_record else 0
        total_impact = qty * unit_cost
        
        # Determine if it's a loss or hold
        bwart = record['BWART']
        if bwart in ['551', '344']:  # Scrap/Write-off
            impact_type = "Write-Off"
        elif bwart == '311':  # Transfer to QA
            impact_type = "On Hold"
        else:
            impact_type = "Other"
        
        scenario_impacts.append({
            'Scenario': scenario_id,
            'Description': record['BKTXT'],
            'Material': material,
            'Plant': plant,
            'Quantity': qty,
            'Unit_Cost': unit_cost,
            'Total_Impact': total_impact,
            'Impact_Type': impact_type
        })

# Display summary table
import pandas as pd
df_summary = pd.DataFrame(scenario_impacts)
display(spark.createDataFrame(df_summary))

# Calculate totals
total_writeoff = sum(s['Total_Impact'] for s in scenario_impacts if s['Impact_Type'] == 'Write-Off')
total_on_hold = sum(s['Total_Impact'] for s in scenario_impacts if s['Impact_Type'] == 'On Hold')

print(f"\n" + "=" * 50)
print(f"TOTAL FINANCIAL IMPACT:")
print(f"  Confirmed Write-Offs: £{total_writeoff:>15,.2f}")
print(f"  Stock on QA Hold:     £{total_on_hold:>15,.2f}")
print(f"                        {'─' * 20}")
print(f"  TOTAL EXPOSURE:       £{total_writeoff + total_on_hold:>15,.2f}")
print(f"=" * 50)

In [ ]:
# CELL: IMPACT VISUALIZATION
import plotly.express as px

if scenario_impacts:
    df_viz = pd.DataFrame(scenario_impacts)
    
    # Bar chart of financial impacts
    fig = px.bar(
        df_viz,
        x='Scenario',
        y='Total_Impact',
        color='Impact_Type',
        text='Description',
        title='Scenario Financial Impact (GBP)',
        color_discrete_map={'Write-Off': '#E74C3C', 'On Hold': '#F39C12', 'Other': '#3498DB'}
    )
    
    fig.update_layout(
        template='plotly_white',
        yaxis_tickformat=',.0f',
        yaxis_tickprefix='£',
        showlegend=True
    )
    
    fig.update_traces(textposition='outside')
    fig.show()

---
## 8. Movement Type Reference
SAP Movement Types used in scenarios

In [ ]:
# CELL: MOVEMENT TYPE REFERENCE

print("=" * 70)
print("SAP MOVEMENT TYPE REFERENCE")
print("=" * 70)

movement_types = [
    ("101", "Goods Receipt", "Receipt of goods from vendor or production"),
    ("201", "Goods Issue - Cost Center", "Consumption to cost center"),
    ("261", "Goods Issue - Production", "Component consumption for production order"),
    ("301", "Plant to Plant Transfer", "Stock transfer between plants"),
    ("311", "Storage Location Transfer", "Transfer between storage locations (e.g., to QA hold)"),
    ("344", "Block Stock Transfer", "Transfer to/from blocked stock (quality issues)"),
    ("551", "Scrapping", "Write-off of damaged/obsolete stock"),
    ("601", "Goods Issue - Delivery", "Stock reduction for customer delivery"),
    ("651", "Returns", "Customer returns to stock"),
]

print(f"\n{'BWART':<8} {'Type':<30} {'Description'}")
print("-" * 70)
for bwart, name, desc in movement_types:
    print(f"{bwart:<8} {name:<30} {desc}")